# Feature Engineering

## Categorisation of bets based on the timing of the match
Timing of bets can be placed into 3 major categories:

1. Before the ban/pick phase
2. After the ban/pick phase and before match has started
3. Anytime before the game has ended

For this EDA, we will focus on the first two categories and the last category can be left as a future extension of the project.

In [2]:
%load_ext autoreload
%autoreload 2
from src.postgresql import get_engine
from src.pipeline.preprocessing import preprocess_df
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)



In [3]:
from src.redis.redis_client import RedisClient

r = RedisClient.get_instance()

In [4]:
engine = get_engine()

In [5]:
df = pd.read_sql(
    "SELECT * FROM pro_matches",
    con=engine
)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101616 entries, 0 to 101615
Data columns (total 28 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   0_hero_id        101600 non-null  float64
 1   1_hero_id        101602 non-null  float64
 2   2_hero_id        101606 non-null  float64
 3   3_hero_id        101602 non-null  float64
 4   4_hero_id        101602 non-null  float64
 5   128_hero_id      101600 non-null  float64
 6   129_hero_id      101604 non-null  float64
 7   130_hero_id      101613 non-null  float64
 8   131_hero_id      101604 non-null  float64
 9   132_hero_id      101608 non-null  float64
 10  0_account_id     101600 non-null  float64
 11  1_account_id     101602 non-null  float64
 12  2_account_id     101606 non-null  float64
 13  3_account_id     101602 non-null  float64
 14  4_account_id     101602 non-null  float64
 15  128_account_id   101600 non-null  float64
 16  129_account_id   101604 non-null  floa

# Feature Selection (Manual)

In [6]:
draft_cols = df.filter(like="_hero_id").columns
player_cols = df.filter(like="_account_id").columns
team_cols = ['radiant_name','dire_name']
label_col = 'radiant_win'
time_col = 'start_time'
uuid_col = 'match_id'

In [7]:
df = preprocess_df(df)
df.info()

Removing 2291 rows with missing values
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99325 entries, 0 to 99324
Data columns (total 25 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   0_hero_id       99325 non-null  object        
 1   1_hero_id       99325 non-null  object        
 2   2_hero_id       99325 non-null  object        
 3   3_hero_id       99325 non-null  object        
 4   4_hero_id       99325 non-null  object        
 5   128_hero_id     99325 non-null  object        
 6   129_hero_id     99325 non-null  object        
 7   130_hero_id     99325 non-null  object        
 8   131_hero_id     99325 non-null  object        
 9   132_hero_id     99325 non-null  object        
 10  0_account_id    99325 non-null  float64       
 11  1_account_id    99325 non-null  float64       
 12  2_account_id    99325 non-null  float64       
 13  3_account_id    99325 non-null  float64       
 14  4_account_id   

In [8]:
df.head()

,0_hero_id,1_hero_id,2_hero_id,3_hero_id,4_hero_id,128_hero_id,129_hero_id,130_hero_id,131_hero_id,132_hero_id,0_account_id,1_account_id,2_account_id,3_account_id,4_account_id,128_account_id,129_account_id,130_account_id,131_account_id,132_account_id,radiant_name,dire_name,radiant_win,start_time,match_id
0,Wraith King,Witch Doctor,Earth Spirit,Leshrac,Dark Seer,Batrider,Lifestealer,Earthshaker,Timbersaw,Grimstroke,452400903.0,294218576.0,493967542.0,455509466.0,413339999.0,116293223.0,173971950.0,101779337.0,194521913.0,278770268.0,Omega Gaming,Incubus Gaming,True,2021-05-17 21:06:51,5999176266
1,Ancient Apparition,Enchantress,Magnus,Ember Spirit,Tiny,Oracle,Storm Spirit,Broodmother,Nyx Assassin,Drow Ranger,86818655.0,916319073.0,126174633.0,88611330.0,203851397.0,97366926.0,97072681.0,71266407.0,184131721.0,107579895.0,Infinity,Crewmates,True,2021-05-17 21:40:35,5999201501
2,Centaur Warrunner,Hoodwink,Razor,Grimstroke,Gyrocopter,Kunkka,Axe,Medusa,Snapfire,Witch Doctor,118207269.0,126469785.0,230580807.0,76600360.0,318588173.0,874542740.0,93159453.0,172739956.0,79818006.0,67893845.0,Wind and Rain,felt,False,2021-05-17 22:02:01,5999214195
3,Void Spirit,Brewmaster,Terrorblade,Snapfire,Abaddon,Rubick,Centaur Warrunner,Ember Spirit,Medusa,Ancient Apparition,874542740.0,93159453.0,172739956.0,79818006.0,67893845.0,126469785.0,118207269.0,230580807.0,318588173.0,76600360.0,felt,Wind and Rain,False,2021-05-17 23:02:25,5999249937
4,Enchantress,Timbersaw,Tiny,Wraith King,Oracle,Faceless Void,Ancient Apparition,Centaur Warrunner,Snapfire,Templar Assassin,126469785.0,118207269.0,230580807.0,318588173.0,76600360.0,874542740.0,67893845.0,93159453.0,79818006.0,172739956.0,Wind and Rain,felt,False,2021-05-18 00:02:06,5999283181


# Feature Engineering

## Create Team Level Features

In [13]:
from collections import deque

In [14]:
def calculate_win_rate(team_histories: list, team_name: str) -> float:
    if not team_histories:
        return 0.5
    
    win = 0
    for match in team_histories:
        if match['radiant_name'] == team_name and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_name and not match['radiant_win']:
            win += 1
        
    return win/ len(team_histories)
        

In [15]:


def update_team_history(team_histories, radiant, dire, match):
    if radiant not in team_histories:
        team_histories[radiant] = deque(maxlen=10)
    if dire not in team_histories:
        team_histories[dire] = deque(maxlen=10)
        
    team_histories[radiant].append(match)
    team_histories[dire].append(match)
    
def update_matchup_history(matchup_histories, radiant, dire, match):
    if (radiant, dire) not in matchup_histories:
        matchup_histories[(radiant, dire)] = deque(maxlen=10)
    
    matchup_histories[(radiant,dire)].append(match)

In [16]:
def calculate_matchup(matchup_histories: list, team_name:str) -> float:
    if not matchup_histories:
        return 0.5
    
    win = 0
    for match in matchup_histories:
        if match['radiant_name'] == team_name and match['radiant_win']:
            win += 1
        elif match['dire_name'] == team_name and not match['radiant_win']:
            win += 1
    
    return win / len(matchup_histories)

In [17]:
team_histories = {}
matchup_histories = {}
team_level_features = []

for _, match in df.iterrows():
    radiant_team = match['radiant_name']
    dire_team = match['dire_name']
    
    # Calculate features for each row
    radiant_win_rate = calculate_win_rate(team_histories.get(radiant_team, []), radiant_team)
    dire_win_rate = calculate_win_rate(team_histories.get(dire_team, []), dire_team)
    
    all_matches = list(matchup_histories.get((radiant_team, dire_team), [])) \
                    + list(matchup_histories.get((dire_team, radiant_team), []))
    matchup_rate = calculate_matchup(all_matches, radiant_team)
    
    # append features to results
    team_level_features.append({
        'match_id': match['match_id'],
        'radiant_win_rate': radiant_win_rate,
        'dire_win_rate': dire_win_rate,
        'radiant_dire_matchup': matchup_rate
    })
    
    # update history dictionaries
    update_team_history(team_histories, radiant_team, dire_team, match)
    update_matchup_history(matchup_histories, radiant_team, dire_team, match)
    
    
team_level_features
    
    

[{'match_id': 5999176266,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999201501,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999214195,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999249937,
  'radiant_win_rate': 1.0,
  'dire_win_rate': 0.0,
  'radiant_dire_matchup': 1.0},
 {'match_id': 5999283181,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999407636,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999433909,
  'radiant_win_rate': 0.5,
  'dire_win_rate': 0.5,
  'radiant_dire_matchup': 0.5},
 {'match_id': 5999441172,
  'radiant_win_rate': 0.0,
  'dire_win_rate': 1.0,
  'radiant_dire_matchup': 0.0},
 {'match_id': 5999477345,
  'radiant_win_rate': 1.0,
  'dire_win_rate': 0.0,
  'radiant_dire_matchup': 1.0},
 {'match_id': 59994

In [1]:
from sqlmodel import Session
from database.schemas.features import TeamFeatures

In [20]:
model_fields = {
    name for name, field in TeamFeatures.__fields__.items()
    if not name.startswith('_')
}

model_fields

{'dire_win_rate', 'match_id', 'radiant_dire_matchup', 'radiant_win_rate'}

In [33]:
# Store to database:

with Session(engine) as session:
    for row in team_level_features:
        # 'row' is a pandas Series, which works similarly to a dictionary
        filtered_data = {
            field: row[field]
            for field in model_fields
            if field in row
        }
        
        # Create the model instance with the filtered data
        team_features = TeamFeatures(**filtered_data)
        session.merge(team_features)
    
    session.commit()

In [47]:
from src.pipeline.features.team_features import TeamFeatureProcessor

obj = TeamFeatureProcessor(r, engine)


In [48]:
obj.clear_history_cache()

Cleared 4962 team history keys from Redis
Cleared 16049 matchup history keys from Redis
Total keys cleared: 21011


In [49]:
data = obj.create_team_features(df)
team_df = pd.DataFrame(data)
team_df

,match_id,radiant_win_rate,dire_win_rate,radiant_dire_matchup
0,5999176266,0.5,0.5,0.500000
1,5999201501,0.5,0.5,0.500000
2,5999214195,0.5,0.5,0.500000
3,5999249937,1.0,0.0,1.000000
4,5999283181,0.5,0.5,0.500000
...,...,...,...,...
99320,8230656847,0.7,0.4,0.500000
99321,8230677659,0.4,0.8,0.400000
99322,8230693148,0.6,0.6,0.625000
99323,8230701740,0.7,0.5,0.500000


In [39]:
team_df_2 = pd.DataFrame(team_level_features)
team_df_2

,match_id,radiant_win_rate,dire_win_rate,radiant_dire_matchup
0,5999176266,0.5,0.5,0.500000
1,5999201501,0.5,0.5,0.500000
2,5999214195,0.5,0.5,0.500000
3,5999249937,1.0,0.0,1.000000
4,5999283181,0.5,0.5,0.500000
...,...,...,...,...
99320,8230656847,0.7,0.4,0.600000
99321,8230677659,0.4,0.8,0.350000
99322,8230693148,0.6,0.6,0.625000
99323,8230701740,0.7,0.5,0.600000


In [50]:
get_dataframe_differences(team_df, team_df_2)

,df1_radiant_dire_matchup,df2_radiant_dire_matchup
1309,0.4,0.363636
1310,0.4,0.416667
1311,0.4,0.461538
1464,0.7,0.727273
1465,0.7,0.666667
...,...,...
99315,0.4,0.450000
99318,0.5,0.450000
99320,0.5,0.600000
99321,0.4,0.350000


## Create hero level feature 

In [7]:
DRAFT_COLS = [
    '0_hero_id', '1_hero_id', '2_hero_id', '3_hero_id', '4_hero_id',
    '128_hero_id', '129_hero_id', '130_hero_id', '131_hero_id', '132_hero_id'
]

In [11]:
selected_cols = DRAFT_COLS + ['match_id']
heroes_features = df[selected_cols]

In [12]:
from sqlmodel import Session
from database.schemas.features import HeroFeatures

In [15]:
with Session(engine) as session:
    for _, row in heroes_features.iterrows():
        # Filter the row data to only include fields in the model
        # Convert to dict first to make it easier to filter
        row_dict = dict(row)
        match_id = row_dict['match_id']
        hero_picks = []
        
        for column, value in row_dict.items():
            if column in DRAFT_COLS:
                hero_picks.append(value)
                
        hero_features = HeroFeatures(
            match_id=match_id,
            hero_picks=hero_picks
        )
        
        session.merge(hero_features)
    
    session.commit()

## Feature Crossing between players and heros



In [51]:
from collections import deque

player_hero_histories = {}
results = []

In [52]:
def get_player_hero_history(account_id, hero_name):
    key = (account_id, hero_name)
    return player_hero_histories.get(key, [])

In [53]:


def calculate_win_rate(account_id, hero_name):
    history = get_player_hero_history(account_id, hero_name)
    if not history:
        return 0.5
    
    return sum(win for win in history) / len(history)

In [54]:
def update_player_hero_histories(account_id, hero_name, histories, win: bool):
    key = (account_id, hero_name)
    if (account_id, hero_name) not in histories:
        histories[key] = deque(maxlen=20)
        
    histories[key].append(win)

In [55]:
for _, match in df.iterrows():
    match_id = match['match_id']
    match_result = {}
    match_result['match_id'] = match_id
    
    radiant_num = range(0, 5)
    dire_num = range(128, 133)
    radiant_win = match['radiant_win']
    
    for i in radiant_num:
        account_id = match[f'{i}_account_id']
        hero_id = match[f'{i}_hero_id']
        # Calculate win rate based on previous matches
        win_rate = calculate_win_rate(account_id, hero_id)
        # Add win rate to results for this match
        match_result[f'player_hero_{i}_win_rate'] = win_rate
        # Update history after calculating
        update_player_hero_histories(account_id, hero_id, player_hero_histories, radiant_win)
    
    for j in dire_num:
        account_id = match[f'{j}_account_id']
        hero_id = match[f'{j}_hero_id']
        dire_win = not radiant_win
        win_rate = calculate_win_rate(account_id, hero_id)
        match_result[f'player_hero_{j}_win_rate'] = win_rate
        update_player_hero_histories(account_id, hero_id, player_hero_histories, dire_win)
    
    # Append complete match result with all player win rates
    results.append(match_result)

results

[{'match_id': 5999176266,
  'player_hero_0_win_rate': 0.5,
  'player_hero_1_win_rate': 0.5,
  'player_hero_2_win_rate': 0.5,
  'player_hero_3_win_rate': 0.5,
  'player_hero_4_win_rate': 0.5,
  'player_hero_128_win_rate': 0.5,
  'player_hero_129_win_rate': 0.5,
  'player_hero_130_win_rate': 0.5,
  'player_hero_131_win_rate': 0.5,
  'player_hero_132_win_rate': 0.5},
 {'match_id': 5999201501,
  'player_hero_0_win_rate': 0.5,
  'player_hero_1_win_rate': 0.5,
  'player_hero_2_win_rate': 0.5,
  'player_hero_3_win_rate': 0.5,
  'player_hero_4_win_rate': 0.5,
  'player_hero_128_win_rate': 0.5,
  'player_hero_129_win_rate': 0.5,
  'player_hero_130_win_rate': 0.5,
  'player_hero_131_win_rate': 0.5,
  'player_hero_132_win_rate': 0.5},
 {'match_id': 5999214195,
  'player_hero_0_win_rate': 0.5,
  'player_hero_1_win_rate': 0.5,
  'player_hero_2_win_rate': 0.5,
  'player_hero_3_win_rate': 0.5,
  'player_hero_4_win_rate': 0.5,
  'player_hero_128_win_rate': 0.5,
  'player_hero_129_win_rate': 0.5,
  'pl

In [56]:
df_results = pd.DataFrame(results)
df_results

,match_id,player_hero_0_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_3_win_rate,player_hero_4_win_rate,player_hero_128_win_rate,player_hero_129_win_rate,player_hero_130_win_rate,player_hero_131_win_rate,player_hero_132_win_rate
0,5999176266,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
1,5999201501,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
2,5999214195,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
3,5999249937,0.500000,0.500000,0.500000,1.00,0.5,0.500000,0.000000,0.500000,0.500000,0.50
4,5999283181,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
...,...,...,...,...,...,...,...,...,...,...,...
99320,8230656847,0.450000,0.538462,0.400000,0.40,0.5,0.384615,0.350000,0.500000,0.600000,0.65
99321,8230677659,0.250000,0.350000,0.625000,0.45,0.6,0.625000,0.600000,1.000000,1.000000,0.00
99322,8230693148,0.777778,0.714286,0.692308,1.00,0.5,0.588235,0.500000,0.600000,0.833333,0.50
99323,8230701740,0.642857,0.500000,0.384615,0.70,0.4,0.300000,0.666667,0.300000,0.550000,0.50


In [61]:
from database.schemas.features import PlayerHeroFeature

In [62]:
model_fields = {
    name for name in PlayerHeroFeature.model_fields.keys()
    if not name.startswith('_')
}

model_fields

{'match_id',
 'player_hero_0_win_rate',
 'player_hero_128_win_rate',
 'player_hero_129_win_rate',
 'player_hero_130_win_rate',
 'player_hero_131_win_rate',
 'player_hero_132_win_rate',
 'player_hero_1_win_rate',
 'player_hero_2_win_rate',
 'player_hero_3_win_rate',
 'player_hero_4_win_rate'}

In [64]:

from sqlmodel import Session

# Create PlayerHeroFeature objects and insert them
with Session(engine) as session:
    # For each match record
    for record in results:
        # Create a new PlayerHeroFeature instance
        player_hero_feature_obj = PlayerHeroFeature(
            match_id=record["match_id"],
            player_hero_0_win_rate=record["player_hero_0_win_rate"],
            player_hero_1_win_rate=record["player_hero_1_win_rate"],
            player_hero_2_win_rate=record["player_hero_2_win_rate"], 
            player_hero_3_win_rate=record["player_hero_3_win_rate"],
            player_hero_4_win_rate=record["player_hero_4_win_rate"],
            player_hero_128_win_rate=record["player_hero_128_win_rate"],
            player_hero_129_win_rate=record["player_hero_129_win_rate"],
            player_hero_130_win_rate=record["player_hero_130_win_rate"],
            player_hero_131_win_rate=record["player_hero_131_win_rate"],
            player_hero_132_win_rate=record["player_hero_132_win_rate"]
        )
        
        # Use merge instead of add
        session.merge(player_hero_feature_obj)
    
    # Commit all records at once
    try:
        session.commit()
        print(f"Successfully stored {len(results)} player-hero feature records")
    except Exception as e:
        session.rollback()
        print(f"Error storing player-hero features: {str(e)}")
            


Successfully stored 99325 player-hero feature records


In [57]:
from src.pipeline.features.player_hero_features import PlayerHeroFeatures

obj = PlayerHeroFeatures(r, engine, 20)
obj.clear_history_cache()
output = obj.create_player_hero_features(df)
output_df = pd.DataFrame(output)


Cleared 165366 player hero histories from redis


In [58]:
output_df

,match_id,player_hero_0_win_rate,player_hero_1_win_rate,player_hero_2_win_rate,player_hero_3_win_rate,player_hero_4_win_rate,player_hero_128_win_rate,player_hero_129_win_rate,player_hero_130_win_rate,player_hero_131_win_rate,player_hero_132_win_rate
0,5999176266,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
1,5999201501,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
2,5999214195,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
3,5999249937,0.500000,0.500000,0.500000,1.00,0.5,0.500000,0.000000,0.500000,0.500000,0.50
4,5999283181,0.500000,0.500000,0.500000,0.50,0.5,0.500000,0.500000,0.500000,0.500000,0.50
...,...,...,...,...,...,...,...,...,...,...,...
99320,8230656847,0.450000,0.538462,0.400000,0.40,0.5,0.384615,0.350000,0.500000,0.600000,0.65
99321,8230677659,0.250000,0.350000,0.625000,0.45,0.6,0.625000,0.600000,1.000000,1.000000,0.00
99322,8230693148,0.777778,0.714286,0.692308,1.00,0.5,0.588235,0.500000,0.600000,0.833333,0.50
99323,8230701740,0.642857,0.500000,0.384615,0.70,0.4,0.300000,0.666667,0.300000,0.550000,0.50


#### Comparing differences between redis v.s. python dictionary to store player_hero_histories

In [59]:
def get_dataframe_differences(df1, df2, df1_name="df1", df2_name="df2"):
    """
    Compare two dataframes and return a DataFrame showing only the differences.
    
    Args:
        df1, df2: DataFrames to compare
        df1_name, df2_name: Names to use in the output for better identification
    
    Returns:
        DataFrame: A DataFrame containing only rows with differences
    """
    if df1.shape != df2.shape or not df1.columns.equals(df2.columns):
        print("Cannot compare: DataFrames have different shapes or columns")
        return None
    
    # Create a mask for rows with any differences
    diff_mask = pd.Series(False, index=df1.index)
    
    # Track which columns have differences
    diff_columns = []
    
    # Compare each column
    for col in df1.columns:
        # Find rows with different values
        comparison = df1[col] != df2[col]
        
        # Handle NaN values specially (NaN != NaN)
        if df1[col].isna().any() or df2[col].isna().any():
            na_mismatch = df1[col].isna() != df2[col].isna()
            comparison = comparison | na_mismatch
            
        # Update the overall mask
        diff_mask = diff_mask | comparison
        
        # If this column has any differences, track it
        if comparison.any():
            diff_columns.append(col)
    
    # If no differences found
    if not diff_mask.any():
        print("DataFrames are identical!")
        return pd.DataFrame()
    
    # Create a combined DataFrame showing differences
    diff_df = pd.DataFrame(index=df1.loc[diff_mask].index)
    
    # Add columns from both DataFrames
    for col in diff_columns:
        diff_df[f"{df1_name}_{col}"] = df1.loc[diff_mask, col]
        diff_df[f"{df2_name}_{col}"] = df2.loc[diff_mask, col]
    
    return diff_df



In [60]:

differences_df = get_dataframe_differences(output_df, df_results, "output_df", "df_results")
differences_df

DataFrames are identical!


""
